In [ ]:
import os
import urllib.request

# Credentials: the bucket name and Application Default Credentials (ADC). The GCS setup script
# writes both to .env; alternatively run `gcloud auth application-default login`.
assert os.getenv("GCS_BUCKET"), "GCS_BUCKET is not set; see 'Setting up GCS' in chapter_1/README.md"
adc_file = os.getenv("GOOGLE_APPLICATION_CREDENTIALS") or os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
assert os.path.exists(adc_file), "No Application Default Credentials found; run `mise run gcs-setup` or `gcloud auth application-default login`"

# The GCS connector is not fetched by Spark's package resolver, so download the shaded jar once.
# Releases: https://github.com/GoogleCloudDataproc/hadoop-connectors/releases
gcs_connector_version = "3.0.4"
local_jar_path = f"./jars/gcs-connector-{gcs_connector_version}-shaded.jar"
if not os.path.exists(local_jar_path):
    os.makedirs("./jars", exist_ok=True)
    url = f"https://github.com/GoogleCloudDataproc/hadoop-connectors/releases/download/v{gcs_connector_version}/gcs-connector-{gcs_connector_version}-shaded.jar"
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, local_jar_path)

In [ ]:
from pyspark.sql import SparkSession

def get_spark_instance_via_adc(catalog_name: str, gcs_bucket: str) -> SparkSession:
    """
    Creates a Spark session configured for reading and writing Iceberg tables on GCS with ADC (Application Default Credentials).

    Args:
        catalog_name (str): The Iceberg catalog name. You can call it whatever you want
        gcs_bucket (str): The GCS bucket we are targeting for our iceberg warehouse

    Returns:
        SparkSession: Configured Spark session instance.
    """
    # Iceberg runtime matching the PySpark version pinned in pyproject.toml (4.0, Scala 2.13)
    spark_version = os.getenv("SPARK_VERSION", "4.0")
    scala_version = os.getenv("SCALA_VERSION", "2.13")
    iceberg_version = os.getenv("ICEBERG_VERSION", "1.10.1")

    iceberg_package = f"org.apache.iceberg:iceberg-spark-runtime-{spark_version}_{scala_version}:{iceberg_version}"

    # Define the Iceberg warehouse path
    warehouse_path = f"gs://{gcs_bucket}/icehouse"

    return SparkSession.builder \
        .appName("local_spark_gcs") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{catalog_name}.type", "hadoop") \
        .config(f"spark.sql.catalog.{catalog_name}.warehouse", warehouse_path) \
        .config("spark.jars.packages", iceberg_package) \
        .config("spark.jars", local_jar_path) \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "false") \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
        .config("spark.driver.host", "localhost") \
        .config("spark.driver.bindAddress", "127.0.0.1") \
        .config("spark.hadoop.google.cloud.auth.type", "APPLICATION_DEFAULT") \
        .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false") \
        .getOrCreate()
    # The last config works around a Hadoop API mismatch between gcs-connector 3.0.x and the
    # Hadoop bundled with PySpark 4.0: with vectored reads on, reading Parquet from GCS fails with
    # NoSuchMethodError: VectoredReadUtils.validateRangeRequest(FileRange).

In [ ]:
catalog_name = "gcs_cat"
spark = get_spark_instance_via_adc(catalog_name, os.getenv("GCS_BUCKET"))

In [ ]:
namespace = "purch_ord"
spark.sql(f"create namespace if not exists {catalog_name}.{namespace}")

In [ ]:
sql = f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{namespace}.orders (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    total_amount DOUBLE
) USING ICEBERG
"""
spark.sql(sql)

In [ ]:
spark.sql(f"insert into {catalog_name}.{namespace}.orders values (1, current_date(), 1001, 250.75)")

In [ ]:
spark.sql(f"select * from {catalog_name}.{namespace}.orders").show()